# Notebook 3 — KiTS23 Radiomics Feature Extraction
**Dataset:** KiTS23 | **Format:** NIfTI (.nii.gz) | **Processes:** Original + Augmented

---

## What This Notebook Does

This notebook extracts radiomics features from KiTS23 NIfTI files using PyRadiomics. It runs on both the original images and the three augmented versions produced by Notebook 1 (eroded, dilated, flipped). Each subset is extracted independently and saved as a separate CSV file. Augmented images are treated as new independent samples — this is the correct workflow as discussed in Notebooks 1 and 2.

**Pipeline position:**
```
[Notebook 1] → [Notebook 2] → [Notebook 3 — YOU ARE HERE] → [Notebook 4]
 KiTS23 Aug     UCSF Aug       KiTS23 Feature Extraction      UCSF Extr
```

**Inputs:**
- Original AML + RCC NIfTI cases
- Augmented AML NIfTI cases from Notebook 1 (eroded / dilated / flipped)
- `pyradiomics_params.yaml` — shared parameter file

**Outputs (one CSV per subset):**
```
CSV/kits23_original_aml.csv
CSV/kits23_original_rcc.csv
CSV/kits23_augmented_eroded.csv
CSV/kits23_augmented_dilated.csv
CSV/kits23_augmented_flipped.csv
```

---

## Why a Shared Parameter File (pyradiomics_params.yaml)

Without a parameter file, PyRadiomics uses defaults that vary with input image properties. Two images with different native spacings produce features computed on different voxel grids — not comparable even if both are from CT scanners.

The shared YAML file enforces three critical settings across both this notebook and Notebook 4:

**Resampling to 1x1x1 mm isotropic:**
KiTS23 native spacing is ~0.78x0.78x3.0 mm. UCSF native spacing is 0.76x0.76x5.0 mm. Without resampling, shape and texture features are computed on geometrically different grids — especially through-plane (3mm vs 5mm). After resampling both to 1mm isotropic, features are computed on the same grid and are directly comparable.

**Bin width fixed at 25 HU:**
Texture features (GLCM, GLRLM, GLSZM) discretize CT intensities into bins. Without a fixed bin width, binning adapts to each image's HU range, producing different matrices for the same tissue. Fixed 25 HU per bin is the renal CT radiomics standard.

**Outlier removal at 3 standard deviations:**
AML contains macroscopic fat (~-100 HU) which creates extreme outliers in the intensity distribution. These can dominate first-order statistics if not removed.

---

## Key Fix vs. Original Pipeline

The original notebook used:
```python
extractor = featureextractor.RadiomicsFeatureExtractor()  # no YAML, all defaults
```

This notebook uses:
```python
extractor = featureextractor.RadiomicsFeatureExtractor('pyradiomics_params.yaml')
```

This is the same YAML used by Notebook 4 for UCSF extraction. Identical configuration across both datasets is what makes the feature CSVs mergeable for ML modeling.

## 1. Install Dependencies

In [ ]:
!pip install SimpleITK nibabel h5py pandas -q
!pip install git+https://github.com/Radiomics/pyradiomics.git -q

## 2. Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import SimpleITK as sitk
from radiomics import featureextractor

print("Libraries loaded.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Initialise Extractor with Shared Parameter File

Upload `pyradiomics_params.yaml` to your Google Drive and set the path below. The same file must be used in Notebook 4.

In [ ]:
YAML_PATH = "/content/drive/MyDrive/pyradiomics_params.yaml"

extractor = featureextractor.RadiomicsFeatureExtractor(YAML_PATH)
print("Extractor initialised with parameter file:", YAML_PATH)
print("Resampling to:", extractor.settings.get('resampledPixelSpacing'))
print("Bin width     :", extractor.settings.get('binWidth'))

## 5. Paths Configuration

In [ ]:
# Original image directories
ORIGINAL_AML_DIR = "/content/drive/MyDrive/kidney-aml/aml/"
ORIGINAL_RCC_DIR = "/content/drive/MyDrive/kits23-test/Test/"   # adjust to your RCC path

# Augmented directories (output of Notebook 1)
AUG_ROOT = "/content/drive/MyDrive/kidney-Augmentation"
AUG_ERODED  = os.path.join(AUG_ROOT, "eroded")
AUG_DILATED = os.path.join(AUG_ROOT, "dilated")
AUG_FLIPPED = os.path.join(AUG_ROOT, "flipped")

# CSV output directory
CSV_OUT = "/content/drive/MyDrive/CSV/kits23/"
os.makedirs(CSV_OUT, exist_ok=True)

## 6. Feature Extraction Function — NIfTI

In [ ]:
def extract_features_nifti(case_dir, extractor, label=None):
    """
    Extracts radiomics features from a single NIfTI case directory.

    Parameters:
        case_dir  : str — path to case folder containing imaging.nii.gz
                    and segmentation.nii.gz
        extractor : RadiomicsFeatureExtractor — initialised with YAML
        label     : str or None — optional label to add as a column

    Returns:
        dict of features, or None if extraction failed
    """
    image_path = os.path.join(case_dir, "imaging.nii.gz")
    mask_path  = os.path.join(case_dir, "segmentation.nii.gz")

    if not os.path.exists(image_path) or not os.path.exists(mask_path):
        print(f"  SKIPPED: missing files in {case_dir}")
        return None

    try:
        # sitk.ReadImage reads real voxel spacing from NIfTI header
        image = sitk.ReadImage(image_path)
        mask  = sitk.ReadImage(mask_path)

        features = dict(extractor.execute(image, mask))
        features["case_id"] = os.path.basename(case_dir)
        if label is not None:
            features["label"] = label
        return features

    except Exception as e:
        print(f"  ERROR in {os.path.basename(case_dir)}: {e}")
        return None


def run_extraction_on_directory(root_dir, extractor, output_csv, label=None):
    """
    Runs feature extraction on all case subdirectories in root_dir.
    Saves results to output_csv.

    Parameters:
        root_dir   : str — directory containing case subfolders
        extractor  : RadiomicsFeatureExtractor
        output_csv : str — full path for output CSV
        label      : str or None — label column value for all cases
    """
    cases    = sorted([c for c in os.listdir(root_dir)
                       if os.path.isdir(os.path.join(root_dir, c))])
    data_rows = []

    print(f"\nExtracting from: {root_dir}")
    print(f"Cases found: {len(cases)}")

    for case in cases:
        case_dir = os.path.join(root_dir, case)
        print(f"  Processing: {case}")
        row = extract_features_nifti(case_dir, extractor, label)
        if row is not None:
            data_rows.append(row)

    if data_rows:
        df = pd.DataFrame(data_rows)
        df.to_csv(output_csv, index=False)
        print(f"  Saved {len(df)} rows → {output_csv}")
    else:
        print(f"  No valid cases extracted.")

    return data_rows

## 7. Run — Original AML Cases

In [ ]:
run_extraction_on_directory(
    root_dir   = ORIGINAL_AML_DIR,
    extractor  = extractor,
    output_csv = os.path.join(CSV_OUT, "kits23_original_aml.csv"),
    label      = "AML"
)

## 8. Run — Original RCC Cases

In [ ]:
run_extraction_on_directory(
    root_dir   = ORIGINAL_RCC_DIR,
    extractor  = extractor,
    output_csv = os.path.join(CSV_OUT, "kits23_original_rcc.csv"),
    label      = "RCC"
)

## 9. Run — Augmented AML Cases (Eroded / Dilated / Flipped)

Each augmented subset is extracted independently and saved to its own CSV. Augmented cases carry the same case_id as their original — patient-level grouping for train/test split must be done in the modeling notebook using case_id.

In [ ]:
for aug_name, aug_dir in [("eroded",  AUG_ERODED),
                           ("dilated", AUG_DILATED),
                           ("flipped", AUG_FLIPPED)]:
    run_extraction_on_directory(
        root_dir   = aug_dir,
        extractor  = extractor,
        output_csv = os.path.join(CSV_OUT, f"kits23_augmented_{aug_name}.csv"),
        label      = "AML"
    )

## 10. Verify Output CSVs

In [ ]:
import pandas as pd

csv_files = sorted([f for f in os.listdir(CSV_OUT) if f.endswith(".csv")])
print(f"{'File':<45} {'Rows':>6} {'Cols':>6}")
print("-" * 60)
for fname in csv_files:
    df = pd.read_csv(os.path.join(CSV_OUT, fname))
    print(f"{fname:<45} {len(df):>6} {len(df.columns):>6}")

# Confirm spacing was applied — diagnostics column should show 1x1x1
print("\n--- Spacing verification (should be [1.0, 1.0, 1.0] after resampling) ---")
sample_csv = os.path.join(CSV_OUT, "kits23_original_aml.csv")
if os.path.exists(sample_csv):
    df = pd.read_csv(sample_csv)
    spacing_col = [c for c in df.columns if "Spacing" in c]
    if spacing_col:
        print(df[spacing_col[0]].unique())